# AEGIS — zokastech — Apache 2.0 / MIT

# Entraînement NER PII — jeux **Hugging Face** publics

Ce notebook enchaîne les mêmes étapes que [`training/README.md`](../../training/README.md) pour les corpus **E3-JSI** et **Ai4Privacy**, puis **fusion**, **fine-tuning**, **publication optionnelle sur le Hub** et **export ONNX**.

1. Jeu **synthétique EU** (base AEGIS, volume réduit pour démo).
2. Téléchargement + conversion IOB2 : [E3-JSI/synthetic-multi-pii-ner-v1](https://huggingface.co/datasets/E3-JSI/synthetic-multi-pii-ner-v1) (MIT).
3. Optionnel : [ai4privacy/pii-masking-300k](https://huggingface.co/datasets/ai4privacy/pii-masking-300k) — **licence** : usage académique encouragé ; usage commercial → voir la fiche HF / `licensing@ai4privacy.com`.
4. **Fusion** des `DatasetDict` sur disque.
5. **`train_ner.py`** (MPS sur Apple Silicon, `--cpu` sur Intel Mac ou si `AEGIS_TRAIN_CPU=1`).
6. **`push_hf_model.py`** (optionnel — `huggingface-cli login` ou `HF_TOKEN`).
7. **`export_onnx.py`**.

**Prérequis** : noyau Python avec `training/` en tête de `sys.path` (cellule ci-dessous). Ne pas coller de vraies PII.

Voir aussi le notebook plus général [`train_ner_pii.ipynb`](train_ner_pii.ipynb) (JSONL locaux).

In [4]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for d in [p, *p.parents]:
        if (d / "training" / "dataset_builder.py").is_file():
            return d
    raise FileNotFoundError("Impossible de trouver la racine du dépôt (training/dataset_builder.py).")


REPO_ROOT = find_repo_root()
TRAINING_DIR = REPO_ROOT / "training"
DATA_DIR = TRAINING_DIR / "data"
OUTPUT_DIR = TRAINING_DIR / "outputs"
EXPORT_DIR = TRAINING_DIR / "exports" / "onnx_ner_hf_public_nb"

os.chdir(REPO_ROOT)
sys.path.insert(0, str(TRAINING_DIR))

print("REPO_ROOT =", REPO_ROOT)
print("TRAINING_DIR =", TRAINING_DIR)

REPO_ROOT = /Users/zouhairkasmi/aegis
TRAINING_DIR = /Users/zouhairkasmi/aegis/training


In [5]:
# %pip install -q -r training/requirements.txt

import subprocess
import sys

print("Python:", sys.executable)

Python: /Users/zouhairkasmi/aegis/.venv/bin/python


In [6]:
# Paramètres partagés par les cellules suivantes
SEED = 42
IMPORT_AI4PRIVACY = True
MAX_AI4PRIVACY = 5_000  # 0 = tout le train HF Ai4Privacy (~178k)

print("SEED =", SEED, "| IMPORT_AI4PRIVACY =", IMPORT_AI4PRIVACY, "| MAX_AI4PRIVACY =", MAX_AI4PRIVACY)

SEED = 42 | IMPORT_AI4PRIVACY = True | MAX_AI4PRIVACY = 5000


## 1. Jeu synthétique EU (IOB2)

Base alignée sur `dataset_builder.LABELS`. Augmentez `NUM_SYNTHETIC` (ex. `50_000`) pour un entraînement sérieux.

In [7]:
from dataset_builder import build_dataset

SEED = 42
NUM_SYNTHETIC = 2_000
SYNTH_PATH = DATA_DIR / "notebook_hf_public_synth"

SYNTH_PATH.parent.mkdir(parents=True, exist_ok=True)
ds_synth = build_dataset(NUM_SYNTHETIC, seed=SEED)
ds_synth.save_to_disk(str(SYNTH_PATH))
print(ds_synth)
print("Sauvegardé:", SYNTH_PATH)

/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 13751.37 examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 1900
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 100
    })
})
Sauvegardé: /Users/zouhairkasmi/aegis/training/data/notebook_hf_public_synth


## 2. Hugging Face → format AEGIS (`import_hf_external_pii.py`)

Le script télécharge les données, mappe les types vers notre schéma IOB2 et enregistre un `DatasetDict` (train / validation).

- **E3-JSI** : `max_samples=0` = intégralité du split (~3k lignes source, ~1,3k exemples retenus après filtrage).
- **Ai4Privacy** : premier téléchargement **lourd** (~centaines de Mo). Pour une démo, gardez `MAX_AI4PRIVACY` modeste (ex. `5000`) ; `0` = tout le split train (~178k lignes).

In [8]:
import subprocess
import sys

E3JSI_PATH = DATA_DIR / "notebook_hf_e3jsi"
AI4_PATH = DATA_DIR / "notebook_hf_ai4privacy"

# E3-JSI (toujours recommandé)
subprocess.check_call(
    [
        sys.executable,
        str(TRAINING_DIR / "import_hf_external_pii.py"),
        "e3jsi",
        "--output",
        str(E3JSI_PATH),
        "--max_samples",
        "0",
        "--val_ratio",
        "0.1",
        "--seed",
        str(SEED),
    ],
    cwd=str(TRAINING_DIR),
)
print("E3-JSI →", E3JSI_PATH)

Chargement E3-JSI/synthetic-multi-pii-ner-v1 …
  … 500 lignes HF, 329 exemples retenus
  … 1000 lignes HF, 552 exemples retenus
  … 1500 lignes HF, 613 exemples retenus
  … 2000 lignes HF, 727 exemples retenus
  … 2500 lignes HF, 976 exemples retenus
Retenu 1339 / 2971 exemples (split=train).


Saving the dataset (1/1 shards): 100%|██████████| 134/134 [00:00<00:00, 28609.66 examples/s]


OK → /Users/zouhairkasmi/aegis/training/data/notebook_hf_e3jsi
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 1205
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 134
    })
})
E3-JSI → /Users/zouhairkasmi/aegis/training/data/notebook_hf_e3jsi


In [9]:
import subprocess
import sys

if IMPORT_AI4PRIVACY:
    cmd = [
        sys.executable,
        str(TRAINING_DIR / "import_hf_external_pii.py"),
        "ai4privacy",
        "--output",
        str(AI4_PATH),
        "--val_ratio",
        "0.05",
        "--seed",
        str(SEED),
    ]
    if MAX_AI4PRIVACY > 0:
        cmd.extend(["--max_samples", str(MAX_AI4PRIVACY)])
    subprocess.check_call(cmd, cwd=str(TRAINING_DIR))
    print("Ai4Privacy →", AI4_PATH)
else:
    print("IMPORT_AI4PRIVACY=False — étape ignorée.")

Chargement ai4privacy/pii-masking-300k (peut être long la 1ère fois) …
  … 250 lignes HF, 212 exemples retenus
  … 500 lignes HF, 426 exemples retenus
  … 750 lignes HF, 646 exemples retenus
  … 1000 lignes HF, 859 exemples retenus
  … 1250 lignes HF, 1075 exemples retenus
  … 1500 lignes HF, 1292 exemples retenus
  … 1750 lignes HF, 1507 exemples retenus
  … 2000 lignes HF, 1713 exemples retenus
  … 2250 lignes HF, 1910 exemples retenus
  … 2500 lignes HF, 2124 exemples retenus
  … 2750 lignes HF, 2337 exemples retenus
  … 3000 lignes HF, 2539 exemples retenus
  … 3250 lignes HF, 2755 exemples retenus
  … 3500 lignes HF, 2970 exemples retenus
  … 3750 lignes HF, 3180 exemples retenus
  … 4000 lignes HF, 3390 exemples retenus
  … 4250 lignes HF, 3591 exemples retenus
  … 4500 lignes HF, 3810 exemples retenus
  … 4750 lignes HF, 4034 exemples retenus
  … 5000 lignes HF, 4258 exemples retenus
Retenu 4258 / 5000 exemples (split=train).


Saving the dataset (1/1 shards): 100%|██████████| 213/213 [00:00<00:00, 39378.80 examples/s]


OK → /Users/zouhairkasmi/aegis/training/data/notebook_hf_ai4privacy
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 4045
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 213
    })
})
Ai4Privacy → /Users/zouhairkasmi/aegis/training/data/notebook_hf_ai4privacy


## 3. Fusion des jeux sur disque

Même format de colonnes (`tokens`, `ner_tags`, `lang`, `domain`) — requis par `merge_hf_datasets.py`.

In [10]:
import subprocess
import sys

MERGED_PATH = DATA_DIR / "notebook_merged_hf_public"

merge_inputs = [str(SYNTH_PATH), str(E3JSI_PATH)]
if IMPORT_AI4PRIVACY:
    merge_inputs.append(str(AI4_PATH))

subprocess.check_call(
    [sys.executable, str(TRAINING_DIR / "merge_hf_datasets.py"), *merge_inputs, "--output", str(MERGED_PATH)],
    cwd=str(TRAINING_DIR),
)
print("Jeu fusionné:", MERGED_PATH)

Fusionné 3 jeux → /Users/zouhairkasmi/aegis/training/data/notebook_merged_hf_public
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 7150
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'lang', 'domain'],
        num_rows: 447
    })
})
Jeu fusionné: /Users/zouhairkasmi/aegis/training/data/notebook_merged_hf_public


Saving the dataset (1/1 shards): 100%|██████████| 447/447 [00:00<00:00, 93123.42 examples/s]


## 4. Fine-tuning (`train_ner.py`)

Sur **Mac Apple Silicon**, MPS partage la RAM système : un **`RuntimeError: MPS backend out of memory`** vient souvent d’ici (souvent à l’**étape AdamW** : `exp_avg_sq`, pas à l’export ONNX). Le notebook utilise un **petit batch** et **`gradient_accumulation_steps`** pour limiter la mémoire.

Réglages par défaut **Darwin** : batch **1**, accum **8**, **`max_seq_length` 256**, **`--adafactor`** (moins de RAM que AdamW sur l’étape optimiseur), gradient checkpointing. En CLI seul : ajoutez **`--adafactor`** à votre commande si AdamW OOM. Si OOM persiste : **`FORCE_CPU = True`** ou `export AEGIS_TRAIN_CPU=1`. `AEGIS_TRAIN_ADAFACTOR=0` désactive Adafactor dans le notebook. Pour une séquence plus longue : `AEGIS_MAX_SEQ_LENGTH=384` si la RAM suit.

Si l’erreur cite **`loss.backward`** (et non plus AdamW), le pic vient de la **rétropropagation**. Avec **« other allocations » ~17 Go**, la marge MPS est quasi nulle : privilégiez **`FORCE_CPU = True`** / `AEGIS_TRAIN_CPU=1` ou libérez de la RAM (navigateur, Docker, autres noyaux). Dernier recours MPS : avant Python, `export AEGIS_RELAX_MPS_MEMORY_CAP=1` (le script `train_ner.py` applique alors `PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0` avant l’import de torch) — risque **swap / gel**.

Sur **CUDA** : pas de `--cpu` ; ajoutez `--fp16` dans la commande si vous l’éditez. **Mac Intel** : `--cpu` est ajouté automatiquement (pas de MPS).

In [11]:
import importlib.util
import os
import platform
import subprocess
import sys

for _pkg in ("seqeval", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg], cwd=str(REPO_ROOT))

MODEL_OUT = OUTPUT_DIR / "ner-hf-public-notebook"
MODEL_OUT.mkdir(parents=True, exist_ok=True)

# Mac + MPS : XLM-R + gros jeu → OOM si batch trop grand. Défauts conservateurs + accumulation.
# Surcharges : AEGIS_TRAIN_BATCH, AEGIS_TRAIN_GRAD_ACCUM (entiers). OOM persistant : FORCE_CPU ou AEGIS_TRAIN_CPU=1.
FORCE_CPU = False
use_cpu_flag = FORCE_CPU or os.environ.get("AEGIS_TRAIN_CPU", "").lower() in ("1", "true", "yes")
if not use_cpu_flag and platform.system() == "Darwin":
    try:
        import torch

        use_cpu_flag = not torch.backends.mps.is_available()
    except ImportError:
        use_cpu_flag = True

# M1 16 Go + MPS : AdamW OOM souvent → Adafactor + seq 256 par défaut. Désactiver Adafactor : AEGIS_TRAIN_ADAFACTOR=0
if platform.system() == "Darwin":
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "1"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "8"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "256"))
else:
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "8"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "1"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "512"))

EVAL_BATCH = min(BATCH, 2) if platform.system() == "Darwin" and not use_cpu_flag else BATCH

train_cmd = [
    sys.executable,
    str(TRAINING_DIR / "train_ner.py"),
    "--dataset",
    str(MERGED_PATH),
    "--output_dir",
    str(MODEL_OUT),
    "--num_train_epochs",
    "1",
    "--per_device_train_batch_size",
    str(BATCH),
    "--per_device_eval_batch_size",
    str(EVAL_BATCH),
    "--gradient_accumulation_steps",
    str(GRAD_ACCUM),
    "--max_seq_length",
    str(MAX_SEQ),
    "--gradient_checkpointing",
]
if use_cpu_flag:
    train_cmd.append("--cpu")
elif platform.system() == "Darwin" and os.environ.get("AEGIS_TRAIN_ADAFACTOR", "1").lower() not in (
    "0",
    "false",
    "no",
):
    train_cmd.append("--adafactor")

print(
    "[train]",
    "CPU (--cpu)" if use_cpu_flag else "MPS ou CUDA",
    "| train_batch =",
    BATCH,
    "| grad_accum =",
    GRAD_ACCUM,
    "| eval_batch =",
    EVAL_BATCH,
    "| adafactor =",
    "--adafactor" in train_cmd,
)
print(" ".join(train_cmd))
subprocess.check_call(train_cmd, cwd=str(TRAINING_DIR))

[train] MPS ou CUDA | train_batch = 1 | grad_accum = 8 | eval_batch = 1 | adafactor = True
/Users/zouhairkasmi/aegis/.venv/bin/python /Users/zouhairkasmi/aegis/training/train_ner.py --dataset /Users/zouhairkasmi/aegis/training/data/notebook_merged_hf_public --output_dir /Users/zouhairkasmi/aegis/training/outputs/ner-hf-public-notebook --num_train_epochs 1 --per_device_train_batch_size 1 --per_device_eval_batch_size 1 --gradient_accumulation_steps 8 --max_seq_length 256 --gradient_checkpointing --adafactor
[train_ner] MPS actif. Si OOM (backward ou AdamW), utilisez --cpu ou libérez de la RAM ; optionnel : AEGIS_RELAX_MPS_MEMORY_CAP=1 (risque swap). Voir training/README.md.


Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 3/894 [00:32<2:37:29, 10.61s/it]Traceback (most recent call last):
  File "/Users/zouhairkasmi/aegis/training/train_ner.py", line 244, in <module>
    main()
  File "/Users/zouhairkasmi/aegis/training/train_ner.py", line 237, in main
    trainer.train()
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/transformers/trainer.py", line 2325, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/zouhairkasmi/aegis/.venv/lib/python3.12/site-packages/transformers/trainer.py", line 2674, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

CalledProcessError: Command '['/Users/zouhairkasmi/aegis/.venv/bin/python', '/Users/zouhairkasmi/aegis/training/train_ner.py', '--dataset', '/Users/zouhairkasmi/aegis/training/data/notebook_merged_hf_public', '--output_dir', '/Users/zouhairkasmi/aegis/training/outputs/ner-hf-public-notebook', '--num_train_epochs', '1', '--per_device_train_batch_size', '1', '--per_device_eval_batch_size', '1', '--gradient_accumulation_steps', '8', '--max_seq_length', '256', '--gradient_checkpointing', '--adafactor']' returned non-zero exit status 1.

## 5. Publication sur le Hugging Face Hub (optionnel)

Après l’entraînement, `MODEL_OUT / "best_hf"` est un checkpoint **Transformers** prêt à être poussé vers [huggingface.co/models](https://huggingface.co/models).

1. Créez un dépôt modèle vide sur le site (ou laissez le script le créer via l’API).
2. Authentification : terminal `huggingface-cli login`, ou variable d’environnement **`HF_TOKEN`** (scope *write*).
3. Renseignez **`HF_REPO_ID`** ci-dessous (`organisation/nom-du-modele`), puis exécutez la cellule. Laissez vide pour **ignorer** cette étape.

La pipeline Rust **aegis-ner** consomme surtout **ONNX** (étape suivante) ; le Hub sert au partage, à la démo `transformers`, ou à ré-entraîner ailleurs.

In [ ]:
import os
import subprocess
import sys

# Ex. "mon-org/aegis-ner-eu-public" ; si vide, lecture de AEGIS_HF_REPO_ID
HF_REPO_ID = ""
if not HF_REPO_ID.strip():
    HF_REPO_ID = os.environ.get("AEGIS_HF_REPO_ID", "").strip()
HF_PRIVATE = os.environ.get("AEGIS_HF_PRIVATE", "").lower() in ("1", "true", "yes")

BEST_HF = MODEL_OUT / "best_hf"
if not HF_REPO_ID:
    print("Aucun HF_REPO_ID / AEGIS_HF_REPO_ID — publication Hub ignorée.")
elif not BEST_HF.is_dir() or not (BEST_HF / "config.json").is_file():
    print("best_hf introuvable — terminez d’abord l’entraînement (section 4).")
else:
    push_cmd = [
        sys.executable,
        str(TRAINING_DIR / "push_hf_model.py"),
        "--model_dir",
        str(BEST_HF),
        "--repo_id",
        HF_REPO_ID,
    ]
    if HF_PRIVATE:
        push_cmd.append("--private")
    print(" ".join(push_cmd))
    subprocess.check_call(push_cmd, cwd=str(TRAINING_DIR))
    print("Hub :", f"https://huggingface.co/{HF_REPO_ID}")

## 6. Export ONNX (`aegis-ner`)

À lancer **après** un entraînement qui a produit `…/best_hf`. Sortie : `model.onnx`, quantifié INT8, `tokenizer_hf/tokenizer.json`.

Si la cellule d’export échoue avec « model_dir introuvable », c’est que l’entraînement (section 4) n’a pas terminé (souvent OOM MPS — voir section 4).

In [10]:
import subprocess
import sys

BEST_HF = MODEL_OUT / "best_hf"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

export_cmd = [
    sys.executable,
    str(TRAINING_DIR / "export_onnx.py"),
    "--model_dir",
    str(BEST_HF),
    "--out_dir",
    str(EXPORT_DIR),
]
print(" ".join(export_cmd))
subprocess.check_call(export_cmd, cwd=str(TRAINING_DIR))
print("Export terminé:", EXPORT_DIR)

/Users/zouhairkasmi/aegis/.venv/bin/python /Users/zouhairkasmi/aegis/training/export_onnx.py --model_dir /Users/zouhairkasmi/aegis/training/outputs/ner-hf-public-notebook/best_hf --out_dir /Users/zouhairkasmi/aegis/training/exports/onnx_ner_hf_public_nb


model_dir introuvable : /Users/zouhairkasmi/aegis/training/outputs/ner-hf-public-notebook/best_hf
Indiquez le dossier du checkpoint Hugging Face (ex. …/outputs/ner-…/best_hf) après train_ner.py.


CalledProcessError: Command '['/Users/zouhairkasmi/aegis/.venv/bin/python', '/Users/zouhairkasmi/aegis/training/export_onnx.py', '--model_dir', '/Users/zouhairkasmi/aegis/training/outputs/ner-hf-public-notebook/best_hf', '--out_dir', '/Users/zouhairkasmi/aegis/training/exports/onnx_ner_hf_public_nb']' returned non-zero exit status 1.

## Suite

- **Hub** : `training/push_hf_model.py` ou variable `AEGIS_HF_REPO_ID` (section 5).
- **Évaluation HTML** : `python training/evaluate.py --dataset … --model_dir …/best_hf --out_report …`.
- **JSONL + HF** : combinez ce notebook avec [`train_ner_pii.ipynb`](train_ner_pii.ipynb) en ajoutant un répertoire issu de `jsonl_to_hf_dataset.py` à la liste `merge_inputs`.
- **Détails des imports HF** : `training/import_hf_external_pii.py`, `training/README.md`.